In [1]:
import os
import json
import re
import time
from unidecode import unidecode
from dotenv import load_dotenv

# Importações do LangChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
load_dotenv(dotenv_path='../.env')

input_path = '../data/reviews_final.json'
output_path = '../data/reviews_analisadas_gemini.json'


def gerar_chave(titulo, ano):
    # Tira acentos e passa para minúsculas
    titulo_limpo = unidecode(str(titulo)).lower()
    # Remove qualquer coisa que não seja letra, número ou espaço
    titulo_limpo = re.sub(r'[^a-z0-9\s]', '', titulo_limpo)
    # Troca espaços por underline e remove espaços em branco sobrando
    titulo_limpo = re.sub(r'\s+', '_', titulo_limpo).strip('_')
    
    # Junta com o ano
    return f"{titulo_limpo}_{ano}"

In [ ]:
# Configurar a estrutura de saída da IA (Parser)
AspectosPermitidos = Literal["Roteiro", "Direção", "Fotografia", "Atuação", "Montagem", "Som", "Arte"]
NiveisPermitidos = Literal["Muito Ruim", "Ruim", "Neutro", "Bom", "Muito Bom"]

class AnaliseDetalhada(BaseModel):
    aspecto: AspectosPermitidos = Field(description="Apenas um dos aspectos listados.")
    ponto_chave: str = Field(description="O que foi observado nas resenhas")
    tipo: Literal["Elogio", "Crítica"] = Field(description="Exatamente 'Elogio' ou 'Crítica'")

class AnaliseCriticaOutput(BaseModel):
    analises: list[AnaliseDetalhada]
    consenso: str = Field(description="Resumo do consenso geral")

class MetricaGrafico(BaseModel):
    aspecto: AspectosPermitidos
    nivel: NiveisPermitidos
    tag_tendencia: str = Field(description="Deve ser exatamento Aspecto_Nivel (ex: Roteiro_Ruim)")

class TagsTendenciasOutput(BaseModel):
    metricas: list[MetricaGrafico]
    sentimento_geral_score: int = Field(description="Nota de 1 a 5")

parser_analise = JsonOutputParser(pydantic_object=AnaliseCriticaOutput)
parser_tags = JsonOutputParser(pydantic_object=TagsTendenciasOutput)

In [ ]:
# Configurar o Modelo Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # melhor opção custo-benefício para análise de texto
    temperature=0.1,
    max_output_tokens=5000
)

In [ ]:
# Criar os Prompts e Pipelines

system_critico = """
## Persona: Você é um Crítico de Cinema e Pesquisador da Cinematografia Nacional.
Sua especialidade é a análise de recepção: você traduz o que o público leigo sente em métricas técnicas de cinema brasileiro.

DIRETRIZES DE EXTRAÇÃO:
1. TRADUÇÃO TÉCNICA: Se o público diz "o filme é lento", identifique se o problema é a 'Montagem' (ritmo) ou 'Roteiro' (falta de pontos de virada).
2. FOCO NACIONAL: Valorize aspectos da produção brasileira nas resenhas (ex: uso de luz natural, regionalismo linguístico, som direto, atuações teatrais vs naturalistas).
3. CATEGORIAS ESTRITAS: Classifique suas observações ÚNICA E EXCLUSIVAMENTE nestes aspectos: [Roteiro, Direção, Fotografia, Atuação, Montagem, Som, Arte].
4. FILTRO DE RUÍDO: Ignore exclamações vazias como "amei" ou "odiei". Busque o substantivo ou a cena que gerou a emoção para fundamentar a análise técnica.
"""

human_critico = """
Tarefa: Realizar uma autópsia crítica das resenhas do filme "{titulo}" ({ano}).

RESENHAS DO PÚBLICO:
{resenhas}

INSTRUÇÕES DE PROCESSO:
1. Leia todas as resenhas e identifique as recorrências (o que 3 ou mais pessoas citaram, seja positivo ou negativo).
2. Para cada recorrência, atribua um dos termos técnicos estritos.
3. Formate sua resposta rigorosamente seguindo as instruções de saída, mantendo a sobriedade acadêmica.

{format_instructions}
"""

prompt_critico = ChatPromptTemplate.from_messages([
    ("system", system_critico),
    ("human", human_critico),
]).partial(format_instructions=parser_analise.get_format_instructions())


system_analista = """
## Persona: Analista de Dados Sênior especializado em Indústria Audiovisual.
Sua tarefa é converter análises textuais subjetivas em métricas técnicas precisas para alimentação de um dashboard de BI.

DIRETRIZES DE PADRONIZAÇÃO:
1. CATEGORIAS FIXAS: Utilize estritamente os aspectos: [Roteiro, Direção, Atuação, Fotografia, Montagem, Som, Arte]. (Não use "Direção de Arte", use "Arte").
2. ESCALA DE VALOR: Para cada aspecto identificado, determine o nível categórico rigorosamente como: [Muito Ruim, Ruim, Neutro, Bom, Muito Bom].
3. SCORE GERAL: Avalie o peso das críticas e elogios para calcular o 'sentimento_geral_score' em uma escala inteira de 1 (Rejeição Total) a 5 (Aclamação Universal).
4. SÍNTESE DE TAGS: A 'tag_tendencia' deve seguir o formato exato 'Aspecto_Nivel' (ex: Fotografia_Ruim, Montagem_Bom).
"""

human_analista = """
Com base na análise crítica abaixo, gere os dados estruturados:

ANÁLISE DO CRÍTICO:
{analise_anterior}

{format_instructions}
"""

prompt_tags = ChatPromptTemplate.from_messages([
    ("system", system_analista),
    ("human", human_analista),
]).partial(format_instructions=parser_tags.get_format_instructions())

# Pipelines
chain_analise = prompt_critico | llm | parser_analise
chain_tags = prompt_tags | llm | parser_tags

def executar_pipeline(dados):
    analise_raw = chain_analise.invoke(dados)
    tags_raw = chain_tags.invoke({"analise_anterior": json.dumps(analise_raw)})
    return {**analise_raw, "metadados_grafico": tags_raw}

In [ ]:
# Carregar dados e processar o loop
with open(input_path, 'r', encoding='utf-8') as f:
    dados_filmes = json.load(f)

resultados = []
total_filmes = len(dados_filmes)

print(f"Iniciando processamento de {total_filmes} filmes...\n")

for i, filme in enumerate(dados_filmes):
    titulo = filme.get('title', 'Sem Título')
    ano = filme.get('release_year', 'Desconhecido')
    resenhas = filme.get('reviews', [])
    qtd_resenhas = len(resenhas)
    
    # Gera a chave única usando a função
    filme['id_movie'] = gerar_chave(titulo, ano)
    
    
    print(f"[{i+1}/{total_filmes}] {titulo} ({ano}) - {qtd_resenhas} resenhas encontradas.")
    
    if qtd_resenhas <= 4:
        print(" -> Pulado: Resenhas insuficientes.")
        filme['analise_gemini'] = None
    else:
        print(" -> Processando com Gemini...")
        try:
            
            texto_resenhas = "\n---\n".join(resenhas)
            
            
            resultado = executar_pipeline({
                "titulo": titulo,
                "ano": ano,
                "resenhas": texto_resenhas
            })
            
            filme['analise_gemini'] = resultado
            print(" -> Sucesso.")
            
        except Exception as e:
            print(f" -> ERRO na API: {e}")
            filme['analise_gemini'] = {"erro": str(e)}
            
        # cooldown de 10 segundos para evitar rate limits
        time.sleep(10) 
        
    # Limpa a lista de resenhas original para o JSON não ficar gigantesco
    filme.pop('reviews', None) 
    
    # Adiciona o filme processado na lista final
    resultados.append(filme)

    # salvar o arquivo final na pasta data
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=4)
    
print(f"\nPipeline finalizado! Arquivo salvo em: {output_path}")

Iniciando processamento de 158 filmes...

[1/158] Ausência (2014) - 15 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[2/158] O Último Cine Drive-in (2014) - 25 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[3/158] Um Homem Só (2015) - 25 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[4/158] O Outro Lado do Paraíso (2014) - 11 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[5/158] Ponto Zero (2016) - 25 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[6/158] O corpo (2015) - 1 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[7/158] O Teto Sobre Nós (2015) - 0 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[8/158] Quando Parei de Me Preocupar com Canalhas (2015) - 19 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[9/158] Bá (2015) - 1 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[10/158] Dá Licença de Contar (2015) - 13 resenhas encontradas.
 -> 

Gemini produced an empty response. Continuing with empty message
Feedback: block_reason=<BlockedReason.PROHIBITED_CONTENT: 'PROHIBITED_CONTENT'> block_reason_message=None safety_ratings=None


 -> ERRO na API: Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
[22/158] O Matador (2017) - 25 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[23/158] Bio (2017) - 6 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[24/158] A Gis (2016) - 10 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[25/158] Tailor (2017) - 4 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[26/158] O Espirito do Bosque (2017) - 0 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[27/158] Telentrega (2017) - 2 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[28/158] Postergados (2016) - 25 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.
[29/158] O Violeiro Fantasma (2017) - 1 resenhas encontradas.
 -> Pulado: Resenhas insuficientes.
[30/158] O Quebra-Cabeça de Sara (2017) - 8 resenhas encontradas.
 -> Processando com Gemini...
 -> Sucesso.


Pipeline finalizado! Arquivo salvo em: ../data/reviews_analisadas_gemini.json
